# GeneFlow AI · Entrenamiento completo en Kaggle

Este notebook entrena **sin supervisión** la mejor configuración de la búsqueda de hiperparámetros con el conjunto de entrenamiento completo, aprovechando una sesión de Kaggle de 12 horas con **dos GPU T4**.

## Cómo lanzarlo

1. En Kaggle: **Create → New Notebook → File → Import Notebook** y sube este archivo.
2. En **Session options**: **Accelerator: GPU T4 ×2** e **Internet: On**.
3. **Save Version → Save & Run All (Commit)**. Puedes cerrar el navegador.
4. Cuando termine, descarga la carpeta **`train`** de la pestaña **Output**.

## Qué entrena

| GPU | Configuración | Origen | Velocidad en T4 | Época completa |
|---|---|---|---|---|
| 0 | **CNN, prueba 12**: `base`, dos bloques, kernel 5, lr 8,2e-4, dropout 0,05 | `reports/search/cnn-best.json`, la mejor de la búsqueda (0,605) | ~405 secuencias/s | ~59 min |
| 1 | CNN, prueba 12 con **otra semilla** | la misma configuración | ~405 secuencias/s | ~59 min |

Cada época recorre **1,4 millones de secuencias** de train, reales y sintéticas, muestreadas con el peso por frecuencia de la búsqueda. La validación usa las **62 mil secuencias** de `val` completas. `test` no se toca.

La segunda GPU entrena la misma configuración con otra semilla. Así se mide la **variabilidad entre ejecuciones**: si las dos acaban muy cerca, las diferencias entre modelos que se vean después son de fiar. Si prefieres aprovecharla para el finalista grande (prueba 20, `large`, 0,597 en la búsqueda), cambia la segunda entrada de `RUNS` por la que está comentada.

## Presupuesto

Las épocas se calculan con el tiempo que queda después de preparar los datos: el 92 % del tiempo restante dividido entre la duración estimada de una época, con un máximo de `MAX_EPOCHS`. Con la sesión completa salen unas **10 épocas**, unos **14 millones de secuencias vistas**, 23 veces más que en la búsqueda. El calendario del learning rate (calentamiento y coseno) se ajusta a ese número de épocas.

**Seguridad.** Cada entrenamiento recibe además como límite estricto el tiempo que queda de sesión menos 30 minutos. Si se acerca, se corta en el lote en curso y conserva los checkpoints de las épocas ya terminadas. Así el notebook siempre acaba a tiempo de guardar las salidas.

## Qué se guarda en `train/<nombre>/`

| Archivo | Contenido |
|---|---|
| `best.pt` | Los pesos de la época con menor pérdida de validación |
| `last.pt` | Los pesos de la última época |
| `history.json` | Pérdida y accuracy por nivel en cada época |
| `run.json` | La configuración completa: hiperparámetros, encoder, cabezas, entrenamiento y número de parámetros |
| `label_space.json` | El espacio de etiquetas con el que se entrenó, necesario para evaluar el modelo |
| `train.log` | El registro completo del entrenamiento |


In [ ]:
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

SESSION_START = time.time()
SESSION_HOURS = 12.0
SAFETY_MARGIN_HOURS = 0.5
EPOCH_BUDGET_FRACTION = 0.92
MAX_EPOCHS = 12

REPOSITORY = "https://github.com/Dexaroz/geneflow-ai-taxonomy-classifier.git"
BRANCH = "main"

WORK = Path("/tmp/geneflow")
REPO = WORK / "repo"
DATA = WORK / "data"
OUTPUT = Path("/kaggle/working/train")
GENEFLOW = REPO / ".venv" / "bin" / "geneflow"
PYTHON = REPO / ".venv" / "bin" / "python"

RUNS = [
    {"name": "cnn-trial12", "params": "reports/search/cnn-best.json", "trial": 12, "seed": 20260923, "sequences_per_second": 405},
    {"name": "cnn-trial12-seed2", "params": "reports/search/cnn-best.json", "trial": 12, "seed": 20260924, "sequences_per_second": 405},
    # {"name": "cnn-trial20", "params": "reports/search/cnn-trials.json", "trial": 20, "seed": 20260923, "sequences_per_second": 301},
]

TRAIN_ARGUMENTS = [
    "--batch-size", "128",
    "--precision", "fp16",
    "--gpu-memory-fraction", "0.92",
]

EVALUATION_SPEEDUP = 3.0
REPORT_EVERY_SECONDS = 600

WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def run(command, **kwargs):
    print("$", " ".join(str(part) for part in command), flush=True)

    return subprocess.run([str(part) for part in command], check=True, **kwargs)


def elapsed_hours():
    return (time.time() - SESSION_START) / 3600


def remaining_hours():
    return SESSION_HOURS - SAFETY_MARGIN_HOURS - elapsed_hours()


print(f"Python del notebook: {sys.version.split()[0]} | trabajo en {WORK} | salidas en {OUTPUT}")

## 1. Entorno: GPU, PyTorch, repositorio y dependencias

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=True,
)
gpus = [line.split(", ") for line in gpu_query.stdout.strip().splitlines()]
driver_major = int(gpus[0][1].split(".")[0])
cuda_index = "cu130" if driver_major >= 580 else "cu126"

for index, (name, driver, memory) in enumerate(gpus):
    print(f"GPU {index}: {name} | driver {driver} | {memory}")

print(f"PyTorch se instalará con {cuda_index}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

if REPO.exists():
    shutil.rmtree(REPO)

run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%h %s"])

run(["uv", "python", "install", "3.14"])
run(["uv", "sync", "--frozen", "--no-default-groups", "--no-install-package", "torch"], cwd=REPO)
run(["uv", "pip", "install", "--python", PYTHON, "torch==2.14.0", "--index-url", f"https://download.pytorch.org/whl/{cuda_index}"], cwd=REPO)

run([PYTHON, "-c", "import torch; print('torch', torch.__version__, '| GPU visibles:', torch.cuda.device_count(), '|', torch.cuda.get_device_name(0))"])

print(f"Entorno listo en {elapsed_hours() * 60:.0f} min")

## 2. Datos: descarga, construcción, datos sintéticos y cachés

In [ ]:
run([GENEFLOW, "prepare", "--data-dir", DATA])
run([GENEFLOW, "cache", "--data-dir", DATA])

train_sequences = json.loads((DATA / "interim" / "geneflow" / "train_tokens.json").read_text())["sequences"]
validation_sequences = json.loads((DATA / "interim" / "geneflow" / "val_tokens.json").read_text())["sequences"]

print(f"Train: {train_sequences:,} secuencias | val: {validation_sequences:,}")
print(f"Datos listos a los {elapsed_hours() * 60:.0f} min de sesión")

## 3. Plan: épocas que caben en el tiempo restante

In [ ]:
active_runs = RUNS[: len(gpus)]


def epoch_minutes(sequences_per_second):
    training = train_sequences / sequences_per_second
    evaluation = validation_sequences / (sequences_per_second * EVALUATION_SPEEDUP)

    return (training + evaluation) / 60


for entry in active_runs:
    minutes = epoch_minutes(entry["sequences_per_second"])
    entry["epochs"] = max(1, min(MAX_EPOCHS, math.floor(remaining_hours() * 60 * EPOCH_BUDGET_FRACTION / minutes)))

    print(
        f"{entry['name']}: {entry['epochs']} épocas de ~{minutes:.0f} min "
        f"(~{entry['epochs'] * minutes / 60:.1f} h, {entry['epochs'] * train_sequences / 1e6:.1f} M secuencias)"
    )

print(f"Tiempo disponible: {remaining_hours():.2f} h")

## 4. Entrenamiento en paralelo, con un informe cada 10 minutos

In [ ]:
def launch(entry, gpu):
    directory = OUTPUT / entry["name"]
    directory.mkdir(parents=True, exist_ok=True)

    environment = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    log = open(directory / "train.log", "a")

    command = [
        GENEFLOW, "train",
        "--data-dir", DATA,
        "--params", REPO / entry["params"],
        "--trial", str(entry["trial"]),
        "--output", directory,
        "--epochs", str(entry["epochs"]),
        "--seed", str(entry["seed"]),
        "--hours", f"{remaining_hours():.3f}",
        *TRAIN_ARGUMENTS,
    ]

    print(f"Lanzando {entry['name']} en la GPU {gpu}: {entry['epochs']} épocas")

    return subprocess.Popen([str(part) for part in command], stdout=log, stderr=subprocess.STDOUT, env=environment)


def progress(entry):
    history_path = OUTPUT / entry["name"] / "history.json"

    if not history_path.exists():
        return "sin épocas terminadas"

    history = json.loads(history_path.read_text())
    last = history[-1]
    accuracy = last["val_accuracy"]

    return (
        f"época {last['epoch']}/{entry['epochs']} | val loss {last['val_loss']:.4f} | "
        f"género {accuracy['genus']:.3f} | especie {accuracy['species']:.3f}"
    )


def last_line(entry):
    path = OUTPUT / entry["name"] / "train.log"
    lines = path.read_text(errors="replace").strip().splitlines() if path.exists() else []

    return lines[-1][-160:] if lines else "(sin salida todavía)"


processes = {entry["name"]: launch(entry, gpu) for gpu, entry in enumerate(active_runs)}

while any(process.poll() is None for process in processes.values()):
    time.sleep(REPORT_EVERY_SECONDS)

    print(f"\n[{elapsed_hours():.2f} h de sesión, quedan {max(0.0, remaining_hours()):.2f} h]")

    for entry in active_runs:
        print(f"  {entry['name']}: {progress(entry)}")
        print(f"    {last_line(entry)}")

for name, process in processes.items():
    print(f"{name} terminó con código {process.returncode}")

print(f"\nEntrenamiento terminado a las {elapsed_hours():.2f} h de sesión")

## 5. Resumen

In [ ]:
levels = ["domain", "kingdom", "phylum", "class", "order", "family", "genus", "species"]

for entry in active_runs:
    directory = OUTPUT / entry["name"]
    history_path = directory / "history.json"

    print(f"\n=== {entry['name']}")

    if not history_path.exists():
        print("  sin épocas terminadas")

        continue

    history = json.loads(history_path.read_text())

    for record in history:
        accuracy = record["val_accuracy"]
        print(
            f"  época {record['epoch']:>2}: train {record['train_loss']:.4f} | val {record['val_loss']:.4f} | "
            + " ".join(f"{level[:4]} {accuracy[level]:.3f}" for level in levels)
            + f" | {record['seconds'] / 60:.0f} min"
        )

    best = min(history, key=lambda record: record["val_loss"])
    objective = (best["val_accuracy"]["genus"] + best["val_accuracy"]["species"]) / 2
    print(f"  MEJOR: época {best['epoch']} | objetivo de la búsqueda {objective:.4f}")

print("\nArchivos para descargar (pestaña Output):")

for path in sorted(OUTPUT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT.parent)} ({path.stat().st_size / 1e6:.1f} MB)")